# Age Classification — Optimised Training Notebook

**Task:** Binary classification — Young (0) vs Old (1) using ResNet-18 (trained from scratch).

**Key optimisations over baseline:**
- Stronger data augmentation (RandomCrop, ColorJitter, RandomErasing, RandomRotation)
- AdamW optimiser with weight decay
- OneCycleLR scheduler for super-convergence
- Label smoothing
- Dropout in classification head
- Gradient clipping
- Validation monitoring with best-checkpoint saving


In [ ]:
import os
import csv
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms

from model_class import build_model

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ── Configuration ──
DATA_ROOT       = "dataset"
TRAIN_DIR       = os.path.join(DATA_ROOT, "train")
VALID_DIR       = os.path.join(DATA_ROOT, "valid")
VALID_CSV       = os.path.join(DATA_ROOT, "valid_labels.csv")

BATCH_SIZE      = 64
NUM_EPOCHS      = 30
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-2
LABEL_SMOOTHING = 0.1
NUM_WORKERS     = 4
IMG_SIZE        = 224

# Set to True for the FINAL submission (trains on train + valid)
INCLUDE_VALID   = False

In [ ]:
# ── Transforms ──
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# ── Datasets ──
class TrainDataset(Dataset):
    """Load images from class sub-folders (train/0/, train/1/)."""
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        for label in [0, 1]:
            cls_dir = os.path.join(root, str(label))
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(cls_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


class ValidDataset(Dataset):
    """Load labelled validation images from a flat directory + CSV."""
    def __init__(self, img_dir, csv_path, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.samples = []
        with open(csv_path) as f:
            for row in csv.DictReader(f):
                self.samples.append((row['image'], int(row['label'])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# Build datasets
train_dataset = TrainDataset(TRAIN_DIR, transform=train_transform)

if INCLUDE_VALID:
    valid_train_ds = ValidDataset(VALID_DIR, VALID_CSV, transform=train_transform)
    combined_dataset = ConcatDataset([train_dataset, valid_train_ds])
    print(f"Training on train + valid: {len(combined_dataset)} images")
else:
    combined_dataset = train_dataset
    print(f"Training on train only: {len(combined_dataset)} images")

train_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

# Validation loader (for monitoring when not in final mode)
val_loader = None
if not INCLUDE_VALID and os.path.isfile(VALID_CSV):
    val_dataset = ValidDataset(VALID_DIR, VALID_CSV, transform=eval_transform)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    print(f"Validation: {len(val_dataset)} images")

In [ ]:
# ── Model, Optimiser, Loss ──
model = build_model(num_classes=2).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                        weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
    anneal_strategy='cos',
)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

In [ ]:
# ── Evaluation helper ──
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = torch.tensor(labels, dtype=torch.long).to(device)
        outputs = model(images)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return correct / total if total > 0 else 0.0

In [ ]:
# ── Training Loop ──
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = torch.tensor(labels, dtype=torch.long).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    train_acc = correct / total
    avg_loss = running_loss / total
    lr_now = optimizer.param_groups[0]['lr']

    msg = f"Epoch {epoch:02d}/{NUM_EPOCHS}  Loss: {avg_loss:.4f}  Train Acc: {train_acc:.4f}  LR: {lr_now:.6f}"

    if val_loader:
        val_acc = evaluate(model, val_loader, DEVICE)
        msg += f"  Val Acc: {val_acc:.4f}"
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model, 'best_model.pth')
            msg += ' *'

    print(msg)

print('\nTraining complete.')
if val_loader and best_val_acc > 0:
    print(f'Best validation accuracy: {best_val_acc:.4f}')

In [ ]:
# ── Save Model ──
# IMPORTANT: save the FULL model, not just state_dict
torch.save(model, 'saved_model.pth')
print(f"Saved to saved_model.pth ({os.path.getsize('saved_model.pth') / 1e6:.1f} MB)")
print("\nFor submission rename:")
print("  model_class.py  ->  roll_no.py")
print("  saved_model.pth ->  roll_no.pth")

In [ ]:
# ── Verify with evaluation script ──
# Uncomment and run after training to verify submission format:
# !python evaluate_submission_student.py \
#     --model_path saved_model.pth \
#     --model_file model_class.py \
#     --data_dir dataset